# 1. Import Library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.autograd import Variable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import easydict
from tqdm.notebook import trange, tqdm
import seaborn as sns

import pickle
import gzip
from dtaidistance import dtw

from scipy import stats
from scipy.stats import pearsonr

# 2. Data prepare

In [2]:
"""
Seed
"""
torch.manual_seed(0)
"""
Model
"""
VAE = 'models/hai_VAE_model_60.pt'
Discriminator_model = 'models/hai_discriminator_model_60.pt'
Generator_model = 'models/hai_generator_model_60.pt'

"""
Config - Mode
"""
args = easydict.EasyDict({
    "Mode": 'Test', 
    "batch_size": 16,
    "device": torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
    "input_size": 22, 
    "latent_size2":100, 
    "latent_size": 100, 
    "output_size": 22, 
    "window_size" : 10, 
    "num_layers": 2,    
    "learning_rate" : 0.001, 
    "max_iter" : 20000, 
    'early_stop' : True,  
    "num" : 0,
})


In [3]:
def minmax_scaler(data):
    numerator=data-np.min(data,0)
    denominator=np.max(data,0)-np.min(data,0)
    return numerator/(denominator+1e-7)

def data_preprocessing(train_input_file, test_input_file):
  
  with gzip.open(train_input_file) as FI:
        train = pickle.load(FI)
  
  train = train[240:, :] 

  with gzip.open(test_input_file) as FI:
        test = pickle.load(FI)

  y = np.where(test[:,-1]==1)

  test = test[:,:-1]

  total = np.concatenate([train, test])
  total=minmax_scaler(total)

  train_final = total[:len(train),:]
  test_final = total[len(train):,:]
  anomal_idx = y[0]
    
  return train_final, test_final, anomal_idx

def Split_data(data):
  interval_n = int(len(data)/10)
  train_data = data[0:interval_n*7] 
  validate_data = data[interval_n*7:] 

  return train_data, validate_data

def make_data_idx(data, window_size):
  input_idx = []
  for idx in range(window_size-1, len(data)):
    input_idx.append(list(range(idx - window_size+1, idx+1)))

  return input_idx

class Get_Dataset(Dataset):

    def __init__(self, data, Windowsize):
      
      self.input_ids = make_data_idx(data, Windowsize)

      self.var_data = np.array(data)
      self.var_data = torch.FloatTensor(self.var_data)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
      temp_input_ids = self.input_ids[idx]
      input_values = self.var_data[temp_input_ids]

      return input_values

In [4]:
# load data
train_data, test_data, anomal_idx = data_preprocessing('../data/hai/hai_normal_60.pickle', '../data/hai/hai_attack_60.pickle')
train_data, validate_data = Split_data(train_data)

train_dataset = Get_Dataset(train_data, args.window_size)
validate_dataset = Get_Dataset(validate_data, args.window_size)
test_dataset = Get_Dataset(test_data, args.window_size)

train_loader = torch.utils.data.DataLoader(
                 dataset=train_dataset,
                 batch_size=args.batch_size,
                 shuffle=True)
valid_loader = torch.utils.data.DataLoader(
                dataset=validate_dataset,
                batch_size=args.batch_size,
                shuffle=False)
test_loader = torch.utils.data.DataLoader(
                dataset=test_dataset,
                batch_size=args.batch_size,
                shuffle=False)

# 3. VAE-GAN Model Training

In [5]:
## 인코더
class Encoder(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.0, bidirectional=False)
        self.hidden2mean = nn.Linear(self.hidden_size*self.num_layers, self.hidden_size)
        self.hidden2logv = nn.Linear(self.hidden_size*self.num_layers, self.hidden_size)

    def encode(self,x):
        mu = self.hidden2mean(x)
        log_var = self.hidden2logv(x)
        return mu, log_var

    def reparametrize(self, mu, logvar):
        std = logvar.mul(0.5).exp_()
        z = torch.randn([self.num_layers, self.batch_size, self.hidden_size]).to(args.device)
        z = z * std + mu
        return z

    def forward(self, x):
        self.batch_size, _, _ = x.size()
        _, (hidden,cell) = self.lstm(x)  # out: tensor of shape (batch_size, seq_length, hidden_size)'
        
        if self.num_layers > 1:
            # flatten hidden state
            hidden = hidden.view(self.batch_size, self.hidden_size*self.num_layers)
        else:
            hidden = hidden.squeeze()

        mu, logvar = self.encode(hidden)
        z = self.reparametrize(mu,logvar)
        
        return mu, logvar, (z, cell)

## Discriminator
class Discriminator(nn.Module):

    def __init__(self, batch_size,input_size, hidden_size, num_layers):
        super().__init__()
        self.batch_size = batch_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,dropout=0.0, bidirectional=False)
        self.fc = nn.Sequential(nn.Linear(hidden_size, 1), nn.Sigmoid())

    def forward(self, x):
        outputs, (hidden, cell) = self.lstm(x)  # out: tensor of shape (batch_size, seq_length, hidden_size)
        output = self.fc(outputs)
        num_dims = len(output.shape)
        reduction_dims = tuple(range(1, num_dims))
        # (batch_size)
        output = torch.mean(output, dim=reduction_dims)
        return output

## Generator
class Generator(nn.Module):

    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(Generator, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.0, bidirectional=False)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(hidden_size, output_size)
        # self.fc = nn.Sequential(nn.Linear(hidden_size, output_size)) ### nn.ReLU

    def forward(self, x, latent):
        output, (_,_) = self.lstm(x, latent)  # out: tensor of shape (batch_size, seq_length, hidden_size)
        prediction = self.fc(output)
        return prediction

def loss_function_VAE(reconstruct_output, batch_data, mu, log_var):
    recon_x = reconstruct_output
    x = batch_data
    mu = mu
    logvar = log_var

    Le = F.mse_loss(recon_x, x)

    KLD_element = mu.pow(2).add_(logvar.exp()).mul_(-1).add_(1).add_(logvar)
    KLD = torch.sum(KLD_element).mul_(-0.5)
    return Le + KLD

def loss_function_discriminator(window_size, output_discriminator_real, output_discriminator_generated, output_discriminator_noise, real_samples_labels, generated_samples_labels):
    loss_discriminator_real = 0
    loss_discriminator_generated = 0

    output_discriminator_generated = torch.stack(output_discriminator_generated, dim=1)
    output_discriminator_real = torch.stack(output_discriminator_real, dim=1)
    output_discriminator_noise = torch.stack(output_discriminator_noise, dim=1)

    output_discriminator_real = torch.clamp(output_discriminator_real, 1e-40, 1.0)
    loss_discriminator_real = -torch.log(output_discriminator_real)

    output_discriminator_generated = torch.clamp(output_discriminator_generated, 1e-40, 1.0)  
    loss_discriminator_generated = -torch.log(1-output_discriminator_generated)
    
    output_discriminator_noise = torch.clamp(output_discriminator_noise, 1e-40, 1.0)  
    loss_discriminator_noise = -torch.log(1-output_discriminator_noise)

    batch_loss = loss_discriminator_real + loss_discriminator_generated + 0.1*loss_discriminator_noise

    return torch.mean(batch_loss)

def loss_function_Generator(reconstruct_output, batch_data, window_size, output_discriminator_generated, output_discriminator_noise, real_samples_labels):
    recon_x = reconstruct_output
    x = batch_data
    
    Le = F.mse_loss(recon_x, x)

    loss_Generator = 0

    output_discriminator_generated = torch.stack(output_discriminator_generated, dim=1)
    output_discriminator_noise = torch.stack(output_discriminator_noise, dim=1)

    output_discriminator_generated = torch.clamp(output_discriminator_generated, 1e-40, 1.0)
    output_discriminator_noise = torch.clamp(output_discriminator_noise, 1e-40, 1.0)
    
    loss_generated = -torch.log(output_discriminator_generated)
    loss_noise = -torch.log(output_discriminator_noise)
    
    loss_Generator = loss_generated + 0.1*loss_noise + Le

    return torch.mean(loss_Generator)

def Model_initialize(input_dim, latent_dim2 ,latent_dim, window_size, num_layers, batch_size):

    discriminator = Discriminator(batch_size=args.batch_size,
                                  input_size=input_dim,
                                  hidden_size=latent_dim,
                                  num_layers=num_layers,)

    generator = Generator(input_size=input_dim,
                          output_size=input_dim,
                          hidden_size=latent_dim2,
                          num_layers=num_layers,
                         )
    encoder = Encoder(input_size=input_dim,
                      hidden_size=latent_dim2,
                      num_layers=num_layers,
                     )
    
    return encoder, discriminator, generator

def run(args, model_VAE, model_discriminator, model_generator, train_loader, test_loader):

    # optimizer 설정
    optimizer_VAE = torch.optim.Adam(model_VAE.parameters(), lr=args.learning_rate)
    optimizer_discriminator = torch.optim.Adam(model_discriminator.parameters(), lr=args.learning_rate)
    optimizer_generator = torch.optim.Adam(model_generator.parameters(), lr=args.learning_rate)

    ## 반복 횟수 Setting
    epochs = tqdm(range(args.max_iter//len(train_loader)+1))

    ## 학습하기
    count = 0
    best_loss_VAE = 100000000
    best_loss_GAN = 100000000

    for epoch in epochs:

        model_VAE.train()
        model_discriminator.train()
        model_generator.train()

        optimizer_VAE.zero_grad()
        optimizer_discriminator.zero_grad()
        optimizer_generator.zero_grad()
        train_iterator = tqdm(enumerate(train_loader), total=len(train_loader), desc="training")

        for i, batch_data in train_iterator:
            batch_data = batch_data.to(args.device)
            batch_size, sequence_length, var_length = batch_data.size()

            real_samples_labels = torch.ones((batch_size, 1))
            generated_samples_labels = torch.zeros((batch_size, 1))

            mu, log_var, encoder_latent = model_VAE(batch_data)

            inv_idx = torch.arange(sequence_length - 1, -1, -1).long()
            output_discriminator_real = []
            output_discriminator_generated = []
            output_discriminator_noise = []
            reconstruct_output = []

            temp_input = torch.zeros((batch_size, 1, var_length), dtype=torch.float).to(batch_data.device)
            hidden = encoder_latent
            
            ########################################################################################################
            z_noise = Variable(torch.zeros((hidden[1].size()[0], hidden[1].size()[1], hidden[1].size()[2]))).cuda() # feature_sizehi
            hidden_noise = Variable(torch.randn((hidden[1].size()[0], hidden[1].size()[1], hidden[1].size()[2]))).cuda() # feature_sizehi

            for t in range(sequence_length):
                temp_input = model_generator(temp_input, hidden)
                temp_output_discriminator_generated = model_discriminator(temp_input)
                temp_output_discriminator_real = model_discriminator(batch_data[:,t,:].unsqueeze(1))
                
                temp_noise = model_generator(temp_input, [z_noise, hidden_noise])
                temp_output_discriminator_noise = model_discriminator(temp_input)

                output_discriminator_real.append(temp_output_discriminator_real)
                output_discriminator_generated.append(temp_output_discriminator_generated)
                output_discriminator_noise.append(temp_output_discriminator_noise)
                reconstruct_output.append(temp_input)

            reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]

            if count > args.max_iter:
                return model_VAE, model_discriminator, model_generator
            count += 1

            batch_data = batch_data.to(args.device)

            loss_VAE = loss_function_VAE(reconstruct_output, batch_data, mu, log_var)
            loss_discriminator = loss_function_discriminator(args.window_size, output_discriminator_real, output_discriminator_generated, output_discriminator_noise, real_samples_labels, generated_samples_labels)
            loss_Generator = loss_function_Generator(reconstruct_output, batch_data, args.window_size, output_discriminator_generated, output_discriminator_noise, real_samples_labels)

            # Backward and optimize
            loss_VAE.backward(retain_graph=True)
            loss_discriminator.backward(retain_graph=True)
            loss_Generator.backward(retain_graph=True)

            optimizer_VAE.step()
            optimizer_discriminator.step()
            optimizer_generator.step()

            optimizer_VAE.zero_grad()
            optimizer_discriminator.zero_grad()
            optimizer_generator.zero_grad()

            train_iterator.set_postfix({
            "loss_VAE": float(loss_VAE),"loss_discriminator": float(loss_discriminator),"loss_Generator": float(loss_Generator)
            })

        model_VAE.eval()
        model_discriminator.eval()
        model_generator.eval()

        eval_loss_VAE = 0
        eval_loss_discriminator = 0
        eval_loss_Generator = 0

        eval_loss = 0

        test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="testing")
        with torch.no_grad():
            for i, batch_data in test_iterator:
                batch_data = batch_data.to(args.device)
                batch_size, sequence_length, var_length = batch_data.size()

                real_samples_labels = torch.ones((batch_size, 1))
                generated_samples_labels = torch.zeros((batch_size, 1))

                mu, log_var, encoder_latent = model_VAE(batch_data)

                inv_idx = torch.arange(sequence_length - 1, -1, -1).long()
                output_discriminator_real = []
                output_discriminator_generated = []
                output_discriminator_noise = []
                reconstruct_output = []

                temp_input = torch.zeros((batch_size, 1, var_length), dtype=torch.float).to(batch_data.device)
                hidden = encoder_latent
                
                ########################################################################################################
                z_noise = Variable(torch.zeros((hidden[1].size()[0], hidden[1].size()[1], hidden[1].size()[2]))).cuda()
                hidden_noise = Variable(torch.randn((hidden[1].size()[0], hidden[1].size()[1], hidden[1].size()[2]))).cuda()

                for t in range(sequence_length):
                    temp_input = model_generator(temp_input, hidden)
                    temp_output_discriminator_generated = model_discriminator(temp_input)
                    temp_output_discriminator_real = model_discriminator(batch_data[:,t,:].unsqueeze(1))
                    
                    temp_noise = model_generator(temp_input, [z_noise, hidden_noise])
                    temp_output_discriminator_noise = model_discriminator(temp_input)

                    output_discriminator_real.append(temp_output_discriminator_real)
                    output_discriminator_generated.append(temp_output_discriminator_generated)
                    output_discriminator_noise.append(temp_output_discriminator_noise)

                    reconstruct_output.append(temp_input)

                reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]

                batch_data = batch_data.to(args.device)

                loss_VAE = loss_function_VAE(reconstruct_output, batch_data, mu, log_var)
                loss_discriminator = loss_function_discriminator(args.window_size, output_discriminator_real, output_discriminator_generated, output_discriminator_noise, real_samples_labels, generated_samples_labels)
                
                loss_Generator = loss_function_Generator(reconstruct_output, batch_data, args.window_size, output_discriminator_generated, output_discriminator_noise, real_samples_labels)
                
                eval_loss_VAE += loss_VAE.mean().item()
                eval_loss_discriminator += loss_discriminator.mean().item()
                eval_loss_Generator += loss_Generator.mean().item()

                test_iterator.set_postfix({
                "eval_loss_VAE": float(loss_VAE),"eval_loss_discriminator": float(loss_discriminator),"eval_loss_Generator": float(loss_Generator),
                })
        eval_loss_VAE = eval_loss_VAE / len(test_loader)
        eval_loss_discriminator = eval_loss_discriminator / len(test_loader)
        eval_loss_Generator = eval_loss_Generator / len(test_loader)
        epochs.set_postfix({
        "Evaluation Score_VAE": float(loss_VAE), "Evaluation Score_discriminator": float(loss_discriminator), "Evaluation Score_Generator": float(loss_Generator),
        })

        if (eval_loss_VAE < best_loss_VAE and eval_loss_discriminator+eval_loss_Generator < best_loss_GAN):

            best_loss_VAE = eval_loss_VAE
            best_loss_GAN = eval_loss_discriminator + eval_loss_Generator

        else:
            if args.early_stop:
                print('early stop condition   best_loss_VAE[{}]  best_loss_GAN[{}]  eval_loss_VAE[{}]  eval_loss_discriminator[{}]  eval_loss_Generator[{}]'
                      .format(best_loss_VAE, best_loss_GAN, eval_loss_VAE,eval_loss_discriminator,eval_loss_Generator))
                return model_VAE, model_discriminator, model_generator

        torch.save(model_VAE, VAE)
        torch.save(model_discriminator, Discriminator_model)
        torch.save(model_generator, Generator_model)
        
    return model_VAE, model_discriminator, model_generator

def discriminate(args, model_discriminator, test_loader):
    test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="testing")
    output_discriminate = []
    with torch.no_grad():
        for i, batch_data in test_iterator:
            batch_data = batch_data.to(args.device)
            batch_size, sequence_length, var_length = batch_data.size()
            temp_output_discriminate = model_discriminator(batch_data)
            temp_output_discriminate = temp_output_discriminate >= torch.FloatTensor([0.5]).to(args.device)
            for i in range(len(temp_output_discriminate)):
                output_discriminate.append(temp_output_discriminate[i])

    return output_discriminate

def reconstruction(args, model_VAE, model_generator, test_loader, num):
    test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="testing")
    loss_list = []
    predictions = []
    true_data = []
    with torch.no_grad():
        for i, batch_data in test_iterator:
            batch_data = batch_data.to(args.device)    
            batch_size, sequence_length, var_length = batch_data.size()
            mu, log_var, encoder_latent = model_VAE(batch_data)
            inv_idx = torch.arange(sequence_length - 1, -1, -1).long()
            reconstruct_output = []

            temp_input = torch.zeros((batch_size, 1, var_length), dtype=torch.float).to(batch_data.device)
            hidden = encoder_latent
            for t in range(sequence_length):
                temp_input = model_generator(temp_input, hidden)
                reconstruct_output.append(temp_input)

            reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]
            batch_data = batch_data.to(args.device)

            for k in range(len(reconstruct_output)):
                predictions.append(reconstruct_output[k][num])
                true_data.append(batch_data[k][num])


    for i in range(len(predictions)):
        Temp_loss_list=abs(predictions[i]-true_data[i]).mean() #50
        loss_list.append(Temp_loss_list)

    return loss_list,predictions,true_data

def Get_Threshold(args, model_VAE, model_generator, model_discriminator, test_loader, num, gamma):
    output_discriminate = discriminate(args, model_discriminator, test_loader)
    loss_list,predictions,true_data = reconstruction(args, model_VAE, model_generator, test_loader, num)
    Res= []
    for i in range(len(loss_list)):
        Res.append(loss_list[i].mean())

    Score = []
    for i in range(len(output_discriminate)):
        Score.append(gamma*(Res[i])+(1-gamma)*output_discriminate[i])
    # sns.distplot(Score, bins=50, kde=True);

    return Score,predictions,true_data

def get_Accuracy(Score, anomal_idx, THRESHOLD, num):
    anomaly_index_true = anomal_idx
    anomaly_index_predict = []
    correct_anomaly = []
    for i in range(0,len(Score)):
        if Score[i] >= THRESHOLD:
            anomaly_index_predict.append(i)

    for i in range(0,len(anomaly_index_predict)):
        k = anomaly_index_predict[i]
        if k in anomaly_index_true:
            correct_anomaly.append(k)

    print(f'Correct anomaly predictions: {len(correct_anomaly)}/{len(anomaly_index_true)}')
    print("Accuracy: {:.2f}%".format(len(correct_anomaly)/len(anomaly_index_true)*100))
    
    return np.array(correct_anomaly) # 

def compare_plot(predictions, true_data, correct_anomaly, uncorrect_normal, num):
    predict=[]
    true=[]
    correct_anomaly_predict=[]
    uncorrect_normal_predict=[]
    
    for i in range(len(predictions)): 
        predict.append(predictions[i][num])
        true.append(true_data[i][num])
  
    for i in correct_anomaly:
        correct_anomaly_predict.append(predict[i])
    for i in uncorrect_normal:
        uncorrect_normal_predict.append(true[i])
    
    # predict = np.array(predict)
    predict = torch.stack(predict)
    # predict = predict.cpu().numpy()
    # true = np.array(true)
    true = torch.stack(true)
    
    correct_anomaly_predict = torch.stack(correct_anomaly_predict)
    uncorrect_normal_predict = torch.stack(uncorrect_normal_predict)
    
    Detect_anomaly_predict = torch.cat([correct_anomaly_predict, uncorrect_normal_predict], dim=0)
    predict_ano = np.concatenate([correct_anomaly, uncorrect_normal])
    
    total_x = len(predict)
    x = np.arange(0,total_x,1)
    plt.figure(figsize=(16, 6))
    plt.plot(x,predict.cpu(),'b',label='Generated value')
    plt.plot(x,true.cpu(),'r',label='True value')
    plt.plot(predict_ano, Detect_anomaly_predict.cpu(),'*g',label='Detected anomaly')
    plt.legend(loc='upper right', fontsize="15")
    plt.ylim([-0.2, 1.5])
    plt.show()

In [6]:
model_VAE, model_discriminator, model_generator = Model_initialize(input_dim=args.input_size, latent_dim2=args.latent_size2, latent_dim=args.latent_size, window_size=args.window_size, num_layers=args.num_layers, batch_size=args.batch_size)
model_VAE.to(args.device)
model_discriminator.to(args.device)
model_generator.to(args.device)

Generator(
  (lstm): LSTM(22, 100, num_layers=2, batch_first=True)
  (relu): ReLU()
  (fc): Linear(in_features=100, out_features=22, bias=True)
)

# 4. Anomaly Detection

In [7]:
## 학습하기
if args.Mode == 'Train':
    model_VAE, model_discriminator, model_generator = run(args, model_VAE, model_discriminator, model_generator, train_loader, valid_loader)

In [8]:
args.Mode = 'Test'

VAE = 'models/hai_VAE_model_60.pt'
Discriminator_model = 'models/hai_discriminator_model_60.pt'
Generator_model = 'models/hai_generator_model_60.pt'

In [9]:
if args.Mode == 'Test':
    model_VAE = torch.load(VAE)
    model_discriminator = torch.load(Discriminator_model)
    model_generator = torch.load(Generator_model)
    model_VAE.to(args.device)
    model_discriminator.to(args.device)
    model_generator.to(args.device)

In [10]:
model_VAE.eval()
model_discriminator.eval()
model_generator.eval()

Generator(
  (lstm): LSTM(22, 100, num_layers=2, batch_first=True)
  (relu): ReLU()
  (fc): Linear(in_features=100, out_features=22, bias=True)
)

In [11]:
Score,predictions,true_data = Get_Threshold(args, model_VAE, model_generator, model_discriminator, test_loader, 0, 1)  

testing:   0%|          | 0/419 [00:00<?, ?it/s]

testing:   0%|          | 0/419 [00:00<?, ?it/s]

In [12]:
THRESHOLD = 0.03
correct_anomaly = get_Accuracy(Score, anomal_idx, THRESHOLD,0)


anomaly_index_true = anomal_idx
correct_anomaly = []
anomaly_index_predict = []


# 예측된 이상치 인덱스 생성
for i in range(0,len(Score)):
    if Score[i] >= THRESHOLD:
        anomaly_index_predict.append(i)

# 정확하게 예측된 이상치 (True Positive)
for i in range(0,len(anomaly_index_predict)):
    k = anomaly_index_predict[i]
    if k in anomaly_index_true:
        correct_anomaly.append(k)
        
# 정확히 예측한 이상치 개수 (TP), 예측에서 실제 이상치가 아닌 것들(FP), 실제인데 예측 안된 것들(FN) 계산
TP = len(correct_anomaly)
FP = len(anomaly_index_predict) - TP
FN = len(anomaly_index_true) - TP

# 지표 계산
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_measure = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

Correct anomaly predictions: 172/185
Accuracy: 92.97%


In [17]:
uncorrect_normal = [65]
detected_anomaly = np.concatenate([correct_anomaly, uncorrect_normal])

In [18]:
print(detected_anomaly)

[  35   36   37   38  239  240  241  242  321  322  363  364  767  768
  879  880  881  882  883  884  885  959 1030 1031 1032 1033 1034 1095
 1096 1202 1203 1204 1205 1206 1328 1329 1353 1354 1355 1356 1357 1443
 1444 1445 1446 1589 1590 1591 1592 1593 1594 1595 1596 1731 1813 1815
 1909 1910 1911 1912 1913 2031 2032 2033 2171 2172 2173 2200 2201 2242
 2243 2244 2245 2361 2362 2546 2637 3531 3532 3533 3622 3623 3624 3625
 3626 3627 3628 3791 3792 3793 3794 3953 3954 3983 3984 4135 4136 4137
 4138 4139 4251 4252 4373 4374 4375 4376 4377 4543 4544 4545 4546 4547
 4605 4606 4607 4608 4609 4737 4738 4739 4740 4741 4845 4846 4847 4848
 4849 5027 5028 5186 5187 5188 5320 5321 5322 5323 5324 5475 5476 5477
 5478 5479 5575 5576 5577 5628 5629 5630 5631 5753 5754 5903 5904 6025
 6026 6071 6072 6073 6074 6544 6545 6546 6547 6548 6590 6591 6592 6631
 6632 6633 6634 6635   65]


# 5. Window Relevance Score Generation

In [19]:
anomalies = detected_anomaly
point_anomaly_list = []
collective_anomaly_list = []

for idx, anomaly in enumerate(anomalies):
    if idx > 0 and anomaly == anomalies[idx - 1] + 1:
        if not collective_anomaly_list or anomalies[idx - 1] != collective_anomaly_list[-1][-1]:
            collective_anomaly_list.append([anomalies[idx - 1]]) 
        collective_anomaly_list[-1].append(anomaly)
    else:
        if idx == 0 or (idx > 0 and anomaly != anomalies[idx - 1] + 1):
            point_anomaly_list.append(anomaly)
point_anomaly_list = [anomaly for anomaly in point_anomaly_list if not any(anomaly in group for group in collective_anomaly_list)]

In [22]:
real_data = true_data.copy()
pre_target_window_list = [i-1 for i in point_anomaly_list]
next_target_window_list = [i+1 for i in point_anomaly_list]
target_list = [[pre, next] for pre, next in zip(pre_target_window_list, next_target_window_list)]

### DTW

In [23]:
import torch
from dtaidistance import dtw
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def mdtw_similarity_window(window1, window2):
    window1 = window1.cpu().numpy()
    window2 = window2.cpu().numpy()

    num_variables = window1.shape[1]  
    dtw_distances = []

    for var in range(num_variables):
        distance = dtw.distance(window1[:, var], window2[:, var])
        dtw_distances.append(distance)

    return np.mean(dtw_distances)

mdtw_scores_list = []
list_num = []

for j in range(len(target_list)):
    mdtw_scores_for_j = []
    
    if len(real_data[target_list[j][0]].shape) == 1:
        window_pre = real_data[target_list[j][0]].unsqueeze(1).clone().detach().to(device)
    else:
        window_pre = real_data[target_list[j][0]].clone().detach().to(device)
    
    if len(real_data[target_list[j][1]].shape) == 1:
        window_next = real_data[target_list[j][1]].unsqueeze(1).clone().detach().to(device)
    else:
        window_next = real_data[target_list[j][1]].clone().detach().to(device)
    
    window1 = torch.cat((window_pre, window_next), dim=0)  
    
    for i in range(len(real_data)):
        if i not in detected_anomaly:
            if len(real_data[i].shape) == 1:
                window2 = real_data[i].unsqueeze(1).clone().detach().to(device)
            else:
                window2 = real_data[i].clone().detach().to(device)

            # MDTW 계산
            score = mdtw_similarity_window(window1, window2)
            mdtw_scores_for_j.append(score)
            list_num.append(i)
    
    mdtw_scores_list.append(mdtw_scores_for_j)
    
# numpy_test = np.array(mdtw_scores_list)
# np.save('./mdtw_scores_list_point', numpy_test)

KeyboardInterrupt: 

In [24]:
score1 = np.load('./mdtw_scores_list_point.npy')

### Pearson

In [25]:
test_loader = torch.utils.data.DataLoader(
                dataset=test_dataset,
                # batch_size=args.batch_size,
                batch_size=1,
                shuffle=False)

correlation_matrices = []
test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="correcting")
for i, batch_data in test_iterator:
    batch_data = batch_data.to(args.device)
    window_np = batch_data.squeeze(0).cpu().numpy()  
    correlation_matrix = np.corrcoef(window_np, rowvar=False)

    raw_data = torch.Tensor.clone(batch_data)
    correlation_matrices.append(correlation_matrix)

correcting:   0%|          | 0/6692 [00:00<?, ?it/s]

c:\Users\jasmi\AppData\Local\anaconda3\envs\cuda2\lib\site-packages\numpy\lib\function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jasmi\AppData\Local\anaconda3\envs\cuda2\lib\site-packages\numpy\lib\function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [26]:
def calculate_similarity(matrix1, matrix2):
    matrix1 = np.nan_to_num(matrix1, nan=0.0)
    matrix2 = np.nan_to_num(matrix2, nan=0.0)
    return np.linalg.norm(matrix1 - matrix2, ord='fro')

pearson_scores_list = [] 
window_num = []  

for j in pre_target_window_list: 
    pearson_scores_for_j = [] 
    window_num_j = [] 
    correlation_matrix1 = correlation_matrices[j]

    for i in range(len(correlation_matrices)):
        if i not in predict_anomaly:  
            correlation_matrix2 = correlation_matrices[i]
            score = calculate_similarity(correlation_matrix1, correlation_matrix2)
            pearson_scores_for_j.append(score) 
            window_num_j.append(i) 
    pearson_scores_list.append(pearson_scores_for_j)
    window_num.append(window_num_j)
    
# numpy_test2 = np.array(pearson_scores_list)
# np.save('./pearson_scores_list_point', numpy_test2)


In [27]:
score2 = np.load('./pearson_scores_list_point.npy')

In [28]:
import torch

attention_matrix = score1 + score2

# attention_matrix 예시 (윈도우 간의 attention score, 텐서로 정의)
attention_matrix = torch.tensor(attention_matrix, dtype=torch.float32)  # 예시 Attention 매트릭스

normalized_attention_matrix = []

# attention_matrix의 최소값과 최대값 계산 후 정규화
for i in range(len(attention_matrix)):
    min_value = attention_matrix[i].min()
    max_value = attention_matrix[i].max()
    # attention_matrix의 값을 0~1 사이로 정규화 (PyTorch 텐서)
    normalized_attention_matrix.append((attention_matrix[i] - min_value) / (max_value - min_value))

# 리스트를 텐서로 변환 (PyTorch 텐서로 변환해야 함)
normalized_attention_matrix = torch.stack(normalized_attention_matrix)

# 각 윈도우에서 상위%에 해당하는 값만 출력
correction_window = []
for i in range(len(normalized_attention_matrix)):
    # 각 윈도우에서 상위%에 해당하는 임계값 계산
    threshold_value = torch.quantile(normalized_attention_matrix[i], 0.999)
    
    print(f"윈도우 {i}의 상위 0.1% 임계값: {threshold_value.item()}")
    window_num_i = []
    for j in range(len(normalized_attention_matrix[i])):
        if normalized_attention_matrix[i][j] > threshold_value:
            print(f"윈도우 {i}, 요소 {j}, 값: {normalized_attention_matrix[i][j].item()}")
            window_num_i.append(window_num[i][j])
    correction_window.append(window_num_i)
print(correction_window)


윈도우 0의 상위 0.1% 임계값: 0.9296688437461853
윈도우 0, 요소 698, 값: 0.9325911402702332
윈도우 0, 요소 4384, 값: 0.9786439538002014
윈도우 0, 요소 4385, 값: 1.0
윈도우 0, 요소 4386, 값: 0.9308567643165588
윈도우 0, 요소 4387, 값: 0.9330527186393738
윈도우 0, 요소 4388, 값: 0.9397823214530945
윈도우 0, 요소 5028, 값: 0.9388808608055115
윈도우 1의 상위 0.1% 임계값: 0.9065853953361511
윈도우 1, 요소 76, 값: 0.9288736581802368
윈도우 1, 요소 1411, 값: 0.9090965390205383
윈도우 1, 요소 1412, 값: 0.9104422330856323
윈도우 1, 요소 4623, 값: 0.913109302520752
윈도우 1, 요소 4624, 값: 0.9185329675674438
윈도우 1, 요소 5048, 값: 1.0
윈도우 1, 요소 5139, 값: 0.9095296859741211
윈도우 2의 상위 0.1% 임계값: 0.9387175440788269
윈도우 2, 요소 698, 값: 1.0
윈도우 2, 요소 1800, 값: 0.9444106221199036
윈도우 2, 요소 3223, 값: 0.9887043833732605
윈도우 2, 요소 3224, 값: 0.9768214225769043
윈도우 2, 요소 3225, 값: 0.9427072405815125
윈도우 2, 요소 4385, 값: 0.9560838341712952
윈도우 2, 요소 5028, 값: 0.9387929439544678
윈도우 3의 상위 0.1% 임계값: 0.9395183324813843
윈도우 3, 요소 698, 값: 1.0
윈도우 3, 요소 1800, 값: 0.9668275713920593
윈도우 3, 요소 1801, 값: 0.941052019596099

# 6. Data correction

In [ ]:
import torch
from tqdm import tqdm

latent_list = []

reconstructed_outputs = []  

for window in correction_window:  
    test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="correcting")
    for i, batch_data in test_iterator:
        if i in window:  
            batch_data = batch_data.to(args.device)
            batch_size, sequence_length, var_length = batch_data.size()

            raw_data = torch.Tensor.clone(batch_data)
            mu, log_var, encoder_latent = model_VAE(batch_data)

            if isinstance(encoder_latent, tuple):
                hidden_state, cell_state = encoder_latent 
            else:
                hidden_state = encoder_latent  
                cell_state = None

            latent_list.append((hidden_state, cell_state))

    hidden_states = [latent[0] for latent in latent_list]  
    cell_states = [latent[1] for latent in latent_list if latent[1] is not None]  
    combined_hidden_state = torch.mean(torch.stack(hidden_states), dim=0)

    if cell_states: 
        combined_cell_state = torch.mean(torch.stack(cell_states), dim=0)
    else:
        combined_cell_state = None

    combined_latent = (combined_hidden_state, combined_cell_state)


    model_generator = model_generator.to(batch_data.device)

    if isinstance(combined_latent, tuple):
        hidden_state, cell_state = combined_latent
        hidden_state = hidden_state.to(batch_data.device)  
        cell_state = cell_state.to(batch_data.device)  
        hidden = (hidden_state, cell_state) 
    else:
        hidden = combined_latent.to(batch_data.device)  

    temp_input = torch.zeros((batch_size, 1, var_length), dtype=torch.float).to(batch_data.device)

    reconstruct_output = []
    inv_idx = torch.arange(sequence_length - 1, -1, -1).long()

    for t in range(sequence_length):
        temp_input = temp_input.to(batch_data.device)
        temp_input = model_generator(temp_input, hidden)
        reconstruct_output.append(temp_input)

    reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]
        reconstructed_outputs.append(reconstruct_output)

correcting: 100%|██████████| 6692/6692 [00:00<00:00, 19590.59it/s]


In [ ]:
test_iterator = tqdm(enumerate(test_loader), total=len(test_loader), desc="correcting")
predictions_v1 = []
true_data_v1 = []

q = torch.tensor([0.1, 0.9]).to(args.device)
lower = []
upper = []
for i in range(len(real_data[0])):
    lower.append(torch.quantile(torch.stack(real_data)[:, i], q)[0])
    upper.append(torch.quantile(torch.stack(real_data)[:, i], q)[1])

with torch.no_grad():

        latent_list = []  
        num = 0
        for i, batch_data in test_iterator:
            if i in point_anomaly_list:
                raw_data = torch.Tensor.clone(batch_data)
                if j in correction_window[num]:
                    num += 1
                    batch_data = batch_data.to(args.device)
                    batch_size, sequence_length, var_length = batch_data.size()
                    mu, log_var, encoder_latent = model_VAE(batch_data)

                    if isinstance(encoder_latent, tuple):
                        hidden_state, cell_state = encoder_latent
                    else:
                        hidden_state = encoder_latent
                        cell_state = None

                    latent_list.append((hidden_state, cell_state))
                    hidden_states = [latent[0] for latent in latent_list]
                    cell_states = [latent[1] for latent in latent_list if latent[1] is not None]
                    combined_hidden_state = torch.mean(torch.stack(hidden_states), dim=0)

                    if cell_states:
                        combined_cell_state = torch.mean(torch.stack(cell_states), dim=0)
                    else:
                        combined_cell_state = None

                    combined_latent = (combined_hidden_state, combined_cell_state)

                    model_generator = model_generator.to(batch_data.device)
                    hidden = (combined_hidden_state.to(batch_data.device),
                            combined_cell_state.to(batch_data.device) if combined_cell_state is not None else None)

                    temp_input = torch.zeros((batch_size, 1, var_length), dtype=torch.float).to(batch_data.device)
                    reconstruct_output = []
                    inv_idx = torch.arange(sequence_length - 1, -1, -1).long()

                    for t in range(sequence_length):
                        temp_input = model_generator(temp_input, hidden)
                        reconstruct_output.append(temp_input)

                    reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]

                for k in range(len(reconstruct_output)):
                    re_dt = reconstruct_output[k][0].to(args.device)
                    for j in range(len(real_data[0])):
                        re_dt[j][re_dt[j] < lower[j]] = lower[j]
                        re_dt[j][re_dt[j] > upper[j]] = upper[j]
                    predictions_v1.append(re_dt)
                    true_data_v1.append(raw_data[k][0].to(args.device))
            else:
            
                predictions_v1.append(real_data[i])
                true_data_v1.append(real_data[i])

predictions_v1 = torch.stack(predictions_v1).to(args.device)
true_data_v1 = torch.stack(true_data_v1).to(args.device)

print(predictions_v1.size())


correcting:   0%|          | 0/6692 [00:00<?, ?it/s]

correcting: 100%|██████████| 6692/6692 [00:00<00:00, 8097.45it/s] 

torch.Size([6692, 22])


In [34]:
# save data to pickle
with gzip.open( "../data/hai_corrected_data.pickle", "wb" ) as file:
    pickle.dump(predictions_v1.cpu().detach().numpy(), file)